## Linear Regression using PCA and Pipeline

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


def generate_sample_data(filepath: str) -> None:
    """
    Helper function to generate a dummy 'input-data.csv' file if one does not exist.
    This ensures the script is self-contained and runnable out-of-the-box.
    """
    np.random.seed(42)
    n_samples = 200

    # Create two correlated features (x1, x2) where PCA is useful
    x1 = np.random.normal(loc=50, scale=10, size=n_samples)
    x2 = 2.5 * x1 + np.random.normal(loc=0, scale=5, size=n_samples)

    # Create a target variable (y) with a linear relationship plus noise
    y = 3.0 * x1 + 1.5 * x2 + np.random.normal(loc=0, scale=15, size=n_samples)

    df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})
    df.to_csv(filepath, index=False)
    print(f"Sample dataset created at '{filepath}'.\n")


def run_pca_regression_pipeline(filepath: str) -> None:
    """
    Loads data from a CSV file, splits it into train/test sets, and runs a
    Scikit-Learn Pipeline containing StandardScaler, PCA, and LinearRegression.
    """
    try:
        # ---------------------------------------------------------------------
        # Step 1: Load the Dataset
        # ---------------------------------------------------------------------
        if not os.path.exists(filepath):
            print(f"'{filepath}' not found. Generating synthetic data for demonstration...")
            generate_sample_data(filepath)

        print(f"Loading data from '{filepath}'...")
        data = pd.read_csv(filepath)

        # Verify that the required columns exist in the dataset
        required_columns = {'x1', 'x2', 'y'}
        if not required_columns.issubset(data.columns):
            raise ValueError(f"CSV must contain the columns: {required_columns}")

        # Separate independent variables (Features: X) and dependent variable (Target: y)
        X = data[['x1', 'x2']]
        y = data['y']

        # ---------------------------------------------------------------------
        # Step 2: Split Data into Training and Testing Sets
        # ---------------------------------------------------------------------
        # CRITICAL: Always split the data BEFORE applying scaling or PCA.
        # Fitting PCA on the whole dataset before splitting causes "data leakage"
        # because information from the test set influences the principal components.
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        print(f"Data split complete: {len(X_train)} training samples, {len(X_test)} testing samples.\n")

        # ---------------------------------------------------------------------
        # Step 3: Define the Scikit-Learn Pipeline
        # ---------------------------------------------------------------------
        # A Pipeline chains multiple processing steps together sequentially.
        # Each step is defined as a tuple: ('step_name', TransformerOrEstimatorObject)
        pipeline = Pipeline(steps=[
            # Step 3a: Standardization (Mean = 0, Variance = 1)
            # PCA is sensitive to the scale of features. Standardizing ensures
            # x1 and x2 contribute equally based on variance rather than magnitude.
            ('scaler', StandardScaler()),

            # Step 3b: Principal Component Analysis (PCA)
            # Reduces dimensionality. Here, n_components=1 projects the 2D data
            # (x1, x2) onto the single 1D axis that captures the maximum variance.
            ('pca', PCA(n_components=1, random_state=42)),

            # Step 3c: Linear Regression Model
            # The final step in a pipeline must be an estimator (predictor).
            # It takes the transformed PCA components as input to predict 'y'.
            ('linear_regression', LinearRegression())
        ])

        # ---------------------------------------------------------------------
        # Step 4: Train (Fit) the Pipeline on the Training Data
        # ---------------------------------------------------------------------
        # Calling .fit() on the pipeline automatically executes:
        # 1. scaler.fit_transform(X_train)
        # 2. pca.fit_transform(scaled_X_train)
        # 3. linear_regression.fit(pca_X_train, y_train)
        print("Training the Pipeline on the training data...")
        pipeline.fit(X_train, y_train)
        print("Pipeline training complete.\n")

        # ---------------------------------------------------------------------
        # Step 5: Inspect Intermediate Pipeline Steps (Optional)
        # ---------------------------------------------------------------------
        # You can access individual steps using the `.named_steps` dictionary
        pca_step = pipeline.named_steps['pca']
        lr_step = pipeline.named_steps['linear_regression']

        explained_variance = pca_step.explained_variance_ratio_[0] * 100
        print(f"PCA Explained Variance Ratio (Component 1): {explained_variance:.2f}%")
        print(f"Linear Regression Coefficient (on PC1): {lr_step.coef_[0]:.4f}")
        print(f"Linear Regression Intercept: {lr_step.intercept_:.4f}\n")

        # ---------------------------------------------------------------------
        # Step 6: Predict Using the Test Data
        # ---------------------------------------------------------------------
        # Calling .predict() on the pipeline automatically executes:
        # 1. scaler.transform(X_test)  <-- Uses mean/std learned from X_train
        # 2. pca.transform(scaled_X_test) <-- Uses eigenvectors learned from X_train
        # 3. linear_regression.predict(pca_X_test)
        print("Generating predictions on the test data...")
        y_pred = pipeline.predict(X_test)

        # ---------------------------------------------------------------------
        # Step 7: Evaluate Model Performance
        # ---------------------------------------------------------------------
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        print("--- Model Evaluation on Test Set ---")
        print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
        print(f"R-squared (R2 Score):           {r2:.4f}\n")

        # Display a side-by-side comparison of the first 5 actual vs predicted values
        comparison_df = pd.DataFrame({
            'Actual y': y_test.values[:5],
            'Predicted y': y_pred[:5]
        })
        print("Sample Predictions (First 5 Test Rows):")
        print(comparison_df.to_string(index=False))

    except FileNotFoundError as fnf_error:
        print(f"File Error: {fnf_error}")
    except ValueError as val_error:
        print(f"Data Validation Error: {val_error}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    # Define the path to the input CSV file
    INPUT_CSV_PATH = "PCA-explanation-temp-input-data.csv"

    # Execute the workflow
    run_pca_regression_pipeline(INPUT_CSV_PATH)

Loading data from 'PCA-explanation-temp-input-data.csv'...
Data split complete: 160 training samples, 40 testing samples.

Training the Pipeline on the training data...
Pipeline training complete.

PCA Explained Variance Ratio (Component 1): 99.06%
Linear Regression Coefficient (on PC1): 44.7218
Linear Regression Intercept: 331.6603

Generating predictions on the test data...
--- Model Evaluation on Test Set ---
Root Mean Squared Error (RMSE): 14.6798
R-squared (R2 Score):           0.9353

Sample Predictions (First 5 Test Rows):
  Actual y  Predicted y
241.599570   234.459097
294.010563   303.178920
273.658176   290.985407
265.424185   258.934270
352.803203   345.912170
